In [3]:
# python/02_data_cleaning.py
# Pulls Query 24 from SQLite, cleans it, saves cleaned_ecommerce_data.csv

import sqlite3
import pandas as pd
import os

DB_PATH  = r"C:\Users\bhuvancw\OneDrive\Desktop\Data Science Projects\DA Projects\shopmart-ecommerce-analytics\data\shopmart.db"
OUT_PATH = r"C:\Users\bhuvancw\OneDrive\Desktop\Data Science Projects\DA Projects\shopmart-ecommerce-analytics\data\processed"

os.makedirs(OUT_PATH, exist_ok=True)

# ── Reusable helpers ──────────────────────────────────────────

def get_connection():
    """Return SQLite connection with performance pragmas set."""
    conn = sqlite3.connect(DB_PATH)
    conn.execute("PRAGMA journal_mode = WAL")
    conn.execute("PRAGMA cache_size   = -64000")
    return conn

def load_master(conn):
    """Pull the full joined master table from SQLite (Query 24)."""
    sql = """
    SELECT
        o.order_id,
        o.order_purchase_timestamp,
        o.order_delivered_customer_date,
        o.order_estimated_delivery_date,
        c.customer_unique_id,
        c.customer_city,
        c.customer_state,
        p.payment_type,
        p.payment_installments,
        p.payment_value                                    AS revenue,
        oi.price                                           AS item_price,
        oi.freight_value,
        COALESCE(t.category_english,
                 pr.product_category_name,'Unknown')       AS category,
        r.review_score,
        CAST(julianday(o.order_delivered_customer_date)
           - julianday(o.order_purchase_timestamp)
           AS INTEGER)                                     AS delivery_days,
        CASE WHEN o.order_delivered_customer_date
                  > o.order_estimated_delivery_date
             THEN 1 ELSE 0 END                             AS was_late,
        strftime('%Y-%m', o.order_purchase_timestamp)      AS order_month,
        CAST(strftime('%Y', o.order_purchase_timestamp) AS INT) AS order_year,
        CAST(strftime('%m', o.order_purchase_timestamp) AS INT) AS month_num,
        CAST(strftime('%H', o.order_purchase_timestamp) AS INT) AS order_hour,
        CASE CAST(strftime('%w', o.order_purchase_timestamp) AS INT)
            WHEN 0 THEN 'Sunday'    WHEN 1 THEN 'Monday'
            WHEN 2 THEN 'Tuesday'   WHEN 3 THEN 'Wednesday'
            WHEN 4 THEN 'Thursday'  WHEN 5 THEN 'Friday'
            WHEN 6 THEN 'Saturday'
        END                                                AS day_of_week
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    LEFT JOIN payments p
        ON o.order_id = p.order_id AND p.payment_sequential = 1
    LEFT JOIN order_items oi
        ON o.order_id = oi.order_id AND oi.order_item_id = 1
    LEFT JOIN products pr    ON oi.product_id = pr.product_id
    LEFT JOIN category_translation t
        ON pr.product_category_name = t.category_portuguese
    LEFT JOIN reviews r ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_purchase_timestamp IS NOT NULL
    ORDER BY o.order_purchase_timestamp
    """
    return pd.read_sql_query(sql, conn)

def fix_dtypes(df):
    """Convert columns to correct data types."""
    df["order_purchase_timestamp"]      = pd.to_datetime(
        df["order_purchase_timestamp"],      errors="coerce")
    df["order_delivered_customer_date"] = pd.to_datetime(
        df["order_delivered_customer_date"], errors="coerce")
    for col in ["revenue","item_price","freight_value",
                "delivery_days","payment_installments","review_score"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

def remove_outliers(df):
    """Remove rows with impossible delivery days."""
    before = len(df)
    df = df[
        df["delivery_days"].isna() |
        ((df["delivery_days"] >= 0) & (df["delivery_days"] <= 200))
    ].copy()
    removed = before - len(df)
    print(f"    Removed {removed} rows with invalid delivery_days")
    return df

def fill_nulls(df):
    """Fill missing categorical values with safe defaults."""
    df["category"]     = df["category"].fillna("Unknown")
    df["payment_type"] = df["payment_type"].fillna("Unknown")
    return df

def add_derived_columns(df):
    """Create extra columns useful for charts and EDA."""
    # Order value bucket for distribution analysis
    df["aov_bucket"] = pd.cut(
        df["revenue"],
        bins=[0,50,100,200,500,1000,99999],
        labels=["<50","50-100","100-200",
                "200-500","500-1000","1000+"]
    )
    return df

def print_summary(df):
    """Print a clean quality summary after cleaning."""
    print(f"\n  Final shape: {df.shape[0]:,} rows × {df.shape[1]} cols")
    print(f"\n  Null % in key columns:")
    for col in ["revenue","delivery_days","review_score",
                "category","customer_state","payment_type"]:
        pct = df[col].isna().mean() * 100
        print(f"    {col:<28} {pct:>5.1f}% null")

def save(df, filename):
    """Save DataFrame to data/processed/."""
    path = os.path.join(OUT_PATH, filename)
    df.to_csv(path, index=False)
    mb = os.path.getsize(path) / 1024 / 1024
    print(f"\n  ✅  Saved: {filename}  ({mb:.1f} MB)")

# ── MAIN ──────────────────────────────────────────────────────
if __name__ == "__main__":
    print("=" * 55)
    print("  STEP 3: Data Cleaning")
    print("=" * 55)

    conn = get_connection()

    print("\n  Loading master table from SQLite...")
    df = load_master(conn)
    print(f"  Raw rows: {len(df):,}")

    print("\n  Cleaning...")
    df = fix_dtypes(df)
    df = remove_outliers(df)
    df = fill_nulls(df)
    df = add_derived_columns(df)

    print_summary(df)
    save(df, "cleaned_ecommerce_data.csv")

    conn.close()
    print("\n  STEP 3 DONE → run step 4: python/03_eda_analysis.py")
    print("=" * 55)

  STEP 3: Data Cleaning

  Loading master table from SQLite...
  Raw rows: 97,007

  Cleaning...
    Removed 2 rows with invalid delivery_days

  Final shape: 97,005 rows × 22 cols

  Null % in key columns:
    revenue                        0.1% null
    delivery_days                  0.0% null
    review_score                   0.7% null
    category                       0.0% null
    customer_state                 0.0% null
    payment_type                   0.0% null

  ✅  Saved: cleaned_ecommerce_data.csv  (21.4 MB)

  STEP 3 DONE → run step 4: python/03_eda_analysis.py
